In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import anndata
import scanpy as sc

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.processed.harmony.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.head()

In [ ]:
adata.obs.index = [x[:-2] for x in adata.obs.index]

In [ ]:
sc.pl.umap(adata, color=['time', 'line', 'assay'], ncols=2)

### per donor, call leiden clusters

In [ ]:
sc.pp.neighbors

In [ ]:
n = 50
d = donors[0]
tmp_adata = adata[adata.obs['line'] == d]
sc.tl.pca(tmp_adata)
sc.external.pp.harmony_integrate(tmp_adata, key='batch')
tmp_adata.obsm['X_pca'] = tmp_adata.obsm['X_pca_harmony']
sc.pp.neighbors(tmp_adata, n_neighbors=n, n_pcs=20, random_state=0)
sc.tl.leiden(tmp_adata, resolution=0.4, random_state=0)
sc.tl.umap(tmp_adata, random_state=0)
sc.pl.umap(tmp_adata, color=['leiden'])

In [ ]:
%%time

n = 50
n_pcs = 20

tmp_results = []
for d in donors:
    tmp_adata = adata[adata.obs['line'] == d]
    sc.tl.pca(tmp_adata)
    sc.external.pp.harmony_integrate(tmp_adata, key='batch')
    tmp_adata.obsm['X_pca'] = tmp_adata.obsm['X_pca_harmony']
    sc.pp.neighbors(tmp_adata, n_neighbors=n, n_pcs=n_pcs, random_state=0)
    sc.tl.leiden(tmp_adata, resolution=0.4, random_state=0)
    sc.tl.umap(tmp_adata, random_state=0)
    sc.pl.umap(tmp_adata, color=['leiden'])
    tmp_adata.obs['umap0'] = tmp_adata.obsm['X_umap'][:,0]
    tmp_adata.obs['umap1'] = tmp_adata.obsm['X_umap'][:,1]
    tmp_results.append(tmp_adata.obs)

In [ ]:
merged_clusts = pd.concat(tmp_results)

In [ ]:
merged_clusts.to_csv(f'{projdir}/csv/integrate_m3c/leiden_clusters.csv', sep='\t')

In [ ]:
!date